In [ ]:
import subprocess
import sys

def install_if_missing(package, import_name=None):
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
install_if_missing('torch')
install_if_missing('torchvision')
install_if_missing('matplotlib')
install_if_missing('numpy')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'✅ GPU Detected: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1000000000.0:.1f} GB')
else:
    DEVICE = torch.device('cpu')
    print('⚠️  No GPU detected — running on CPU (all operations remain functional)')
print(f'   PyTorch: {torch.__version__}')
print(f'   Device:  {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
@dataclass
class DomainConfig:
    name: str
    image_size: Tuple[int, int]
    num_objects_range: Tuple[int, int]
    object_size_range: Tuple[float, float]
    latency_budget_ms: float
    required_precision: float
    scene_complexity: float
    num_classes: int
    description: str
DOMAIN_CONFIGS = {'autonomous': DomainConfig(name='Autonomous / Real-Time', image_size=(320, 320), num_objects_range=(3, 8), object_size_range=(0.05, 0.25), latency_budget_ms=15.0, required_precision=0.5, scene_complexity=0.3, num_classes=5, description='Vehicles, pedestrians, traffic signs — speed-critical'), 'medical': DomainConfig(name='Medical Imaging', image_size=(512, 512), num_objects_range=(1, 4), object_size_range=(0.02, 0.1), latency_budget_ms=200.0, required_precision=0.95, scene_complexity=0.5, num_classes=3, description='Tumors, lesions, nodules — precision-critical'), 'remote_sensing': DomainConfig(name='Remote Sensing / Aerial', image_size=(640, 640), num_objects_range=(5, 15), object_size_range=(0.01, 0.15), latency_budget_ms=100.0, required_precision=0.7, scene_complexity=0.9, num_classes=7, description='Vehicles, buildings, ships in aerial/satellite imagery — context-critical')}
MAX_NUM_CLASSES = max((cfg.num_classes for cfg in DOMAIN_CONFIGS.values()))
print('📋 Domain Configurations Loaded:')
print('=' * 70)
for key, cfg in DOMAIN_CONFIGS.items():
    print(f'  [{key.upper()}] {cfg.name}')
    print(f'    Resolution:    {cfg.image_size}')
    print(f'    Latency budget:{cfg.latency_budget_ms:.0f} ms')
    print(f'    Precision req: {cfg.required_precision:.2f}')
    print(f'    Complexity:    {cfg.scene_complexity:.2f}')
    print(f'    → {cfg.description}')
    print()

class MultiDomainDetectionDataset(torch.utils.data.Dataset):

    def __init__(self, domain_key: str, num_samples: int=50):
        self.config = DOMAIN_CONFIGS[domain_key]
        self.domain_key = domain_key
        self.num_samples = num_samples
        self.rng = np.random.RandomState(hash(domain_key) % 2 ** 31)

    def __len__(self) -> int:
        return self.num_samples

    def _generate_domain_image(self, h: int, w: int) -> torch.Tensor:
        if self.domain_key == 'autonomous':
            img = torch.zeros(3, h, w)
            for row in range(h // 2):
                ratio = row / (h // 2)
                img[0, row, :] = 0.3 + 0.2 * ratio
                img[1, row, :] = 0.4 + 0.2 * ratio
                img[2, row, :] = 0.7 - 0.1 * ratio
            for row in range(h // 2, h):
                ratio = (row - h // 2) / (h // 2)
                img[0, row, :] = 0.3 + 0.15 * ratio
                img[1, row, :] = 0.3 + 0.15 * ratio
                img[2, row, :] = 0.3 + 0.1 * ratio
            img += torch.randn_like(img) * 0.05
        elif self.domain_key == 'medical':
            base = torch.randn(1, h, w) * 0.15 + 0.4
            yy, xx = torch.meshgrid(torch.linspace(-1, 1, h), torch.linspace(-1, 1, w), indexing='ij')
            body_mask = (xx ** 2 + yy ** 2 < 0.7).float() * 0.2
            base = base + body_mask.unsqueeze(0)
            img = base.repeat(3, 1, 1)
            img[0] *= 0.95
            img += torch.randn_like(img) * 0.03
        elif self.domain_key == 'remote_sensing':
            img = torch.zeros(3, h, w)
            block_size = max(h // 8, 1)
            for i in range(0, h, block_size):
                for j in range(0, w, block_size):
                    color = torch.rand(3) * 0.4 + 0.2
                    end_i = min(i + block_size, h)
                    end_j = min(j + block_size, w)
                    for c in range(3):
                        img[c, i:end_i, j:end_j] = color[c]
            img += torch.randn_like(img) * 0.08
        else:
            img = torch.rand(3, h, w)
        return img.clamp(0, 1)

    def __getitem__(self, idx: int) -> Dict:
        cfg = self.config
        h, w = cfg.image_size
        image = self._generate_domain_image(h, w)
        num_objs = self.rng.randint(cfg.num_objects_range[0], cfg.num_objects_range[1] + 1)
        boxes = []
        labels = []
        for _ in range(num_objs):
            obj_w = self.rng.uniform(cfg.object_size_range[0], cfg.object_size_range[1])
            obj_h = self.rng.uniform(cfg.object_size_range[0], cfg.object_size_range[1])
            cx = self.rng.uniform(obj_w / 2, 1.0 - obj_w / 2)
            cy = self.rng.uniform(obj_h / 2, 1.0 - obj_h / 2)
            x1 = max(0.0, cx - obj_w / 2)
            y1 = max(0.0, cy - obj_h / 2)
            x2 = min(1.0, cx + obj_w / 2)
            y2 = min(1.0, cy + obj_h / 2)
            boxes.append([x1, y1, x2, y2])
            labels.append(self.rng.randint(0, cfg.num_classes))
            px1, py1 = (int(x1 * w), int(y1 * h))
            px2, py2 = (int(x2 * w), int(y2 * h))
            obj_color = torch.rand(3) * 0.3 + 0.5
            for c in range(3):
                image[c, py1:py2, px1:px2] = obj_color[c] + torch.randn(py2 - py1, px2 - px1) * 0.05
        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
        labels_tensor = torch.tensor(labels, dtype=torch.long)
        metadata = torch.tensor([cfg.latency_budget_ms / 200.0, cfg.required_precision, cfg.scene_complexity], dtype=torch.float32)
        metadata += torch.randn(3) * 0.02
        metadata = metadata.clamp(0, 1)
        return {'image': image.clamp(0, 1), 'boxes': boxes_tensor, 'labels': labels_tensor, 'metadata': metadata, 'domain': self.domain_key}

def collate_fn(batch: List[Dict]) -> Dict:
    return {'image': torch.stack([b['image'] for b in batch]), 'boxes': [b['boxes'] for b in batch], 'labels': [b['labels'] for b in batch], 'metadata': torch.stack([b['metadata'] for b in batch]), 'domain': [b['domain'] for b in batch]}
print('🗂️  Creating datasets...')
datasets = {}
for domain_key in DOMAIN_CONFIGS:
    ds = MultiDomainDetectionDataset(domain_key, num_samples=30)
    datasets[domain_key] = ds
    sample = ds[0]
    print(f"  [{domain_key}] image={sample['image'].shape}, boxes={sample['boxes'].shape}, metadata={sample['metadata'].tolist()}")
print('\n✅ All domain datasets created successfully.')
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
domain_keys = list(DOMAIN_CONFIGS.keys())
for ax, key in zip(axes, domain_keys):
    sample = datasets[key][0]
    img = sample['image'].permute(1, 2, 0).numpy()
    ax.imshow(img)
    h, w = (sample['image'].shape[1], sample['image'].shape[2])
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE']
    for i, box in enumerate(sample['boxes']):
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle((x1 * w, y1 * h), (x2 - x1) * w, (y2 - y1) * h, linewidth=2, edgecolor=colors[i % len(colors)], facecolor='none')
        ax.add_patch(rect)
        ax.text(x1 * w, y1 * h - 3, f"cls:{sample['labels'][i].item()}", fontsize=8, color=colors[i % len(colors)], fontweight='bold', backgroundcolor='black')
    cfg = DOMAIN_CONFIGS[key]
    ax.set_title(f'{cfg.name}\n{cfg.image_size[0]}x{cfg.image_size[1]} | Latency<={cfg.latency_budget_ms:.0f}ms | Precision={cfg.required_precision:.2f}', fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle('Simulated Multi-Domain Detection Samples', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('domain_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Domain samples visualized.')

In [ ]:
class SharedBackbone(nn.Module):

    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x

class OneStageHead(nn.Module):

    def __init__(self, in_channels: int=256, num_classes: int=10, max_detections: int=50):
        super().__init__()
        self.num_classes = num_classes
        self.max_detections = max_detections
        self.conv_tower = nn.Sequential(nn.Conv2d(in_channels, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.cls_head = nn.Conv2d(128, num_classes, 3, padding=1)
        self.reg_head = nn.Conv2d(128, 4, 3, padding=1)

    def forward(self, features: torch.Tensor) -> Dict[str, torch.Tensor]:
        B = features.shape[0]
        tower_out = self.conv_tower(features)
        cls_logits = self.cls_head(tower_out)
        reg_preds = self.reg_head(tower_out)
        cls_logits = cls_logits.permute(0, 2, 3, 1).reshape(B, -1, self.num_classes)
        reg_preds = reg_preds.permute(0, 2, 3, 1).reshape(B, -1, 4)
        cls_probs = torch.softmax(cls_logits, dim=-1)
        scores, labels = cls_probs.max(dim=-1)
        boxes = torch.sigmoid(reg_preds)
        K = min(self.max_detections, scores.shape[1])
        topk_scores, topk_idx = scores.topk(K, dim=1)
        topk_labels = labels.gather(1, topk_idx)
        topk_boxes = boxes.gather(1, topk_idx.unsqueeze(-1).expand(-1, -1, 4))
        return {'boxes': topk_boxes, 'scores': topk_scores, 'labels': topk_labels}

class TwoStageHead(nn.Module):

    def __init__(self, in_channels: int=256, num_classes: int=10, num_proposals: int=64, roi_size: int=7):
        super().__init__()
        self.num_classes = num_classes
        self.num_proposals = num_proposals
        self.roi_size = roi_size
        self.rpn_conv = nn.Sequential(nn.Conv2d(in_channels, 256, 3, padding=1), nn.ReLU(inplace=True))
        self.rpn_cls = nn.Conv2d(256, 1, 1)
        self.rpn_reg = nn.Conv2d(256, 4, 1)
        roi_feat_dim = in_channels * roi_size * roi_size
        self.roi_fc = nn.Sequential(nn.Linear(roi_feat_dim, 512), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(512, 256), nn.ReLU(inplace=True))
        self.roi_cls = nn.Linear(256, num_classes)
        self.roi_reg = nn.Linear(256, 4)

    def forward(self, features: torch.Tensor) -> Dict[str, torch.Tensor]:
        B, C, fH, fW = features.shape
        rpn_feat = self.rpn_conv(features)
        rpn_scores = torch.sigmoid(self.rpn_cls(rpn_feat))
        rpn_boxes = torch.sigmoid(self.rpn_reg(rpn_feat))
        rpn_scores_flat = rpn_scores.reshape(B, -1)
        rpn_boxes_flat = rpn_boxes.permute(0, 2, 3, 1).reshape(B, -1, 4)
        K = min(self.num_proposals, rpn_scores_flat.shape[1])
        _, topk_idx = rpn_scores_flat.topk(K, dim=1)
        proposals = rpn_boxes_flat.gather(1, topk_idx.unsqueeze(-1).expand(-1, -1, 4))
        roi_features = []
        for b in range(B):
            batch_rois = []
            for k in range(K):
                x1, y1, x2, y2 = proposals[b, k]
                fx1 = int((x1 * fW).clamp(0, fW - 1).item())
                fy1 = int((y1 * fH).clamp(0, fH - 1).item())
                fx2 = int((x2 * fW).clamp(fx1 + 1, fW).item())
                fy2 = int((y2 * fH).clamp(fy1 + 1, fH).item())
                roi_region = features[b:b + 1, :, fy1:fy2, fx1:fx2]
                if roi_region.numel() == 0:
                    roi_region = features[b:b + 1, :, :1, :1]
                pooled = F.adaptive_avg_pool2d(roi_region, (self.roi_size, self.roi_size))
                batch_rois.append(pooled.flatten())
            roi_features.append(torch.stack(batch_rois))
        roi_features = torch.stack(roi_features).to(features.device)
        roi_out = self.roi_fc(roi_features)
        cls_logits = self.roi_cls(roi_out)
        refined_boxes = torch.sigmoid(self.roi_reg(roi_out))
        cls_probs = torch.softmax(cls_logits, dim=-1)
        scores, labels = cls_probs.max(dim=-1)
        return {'boxes': refined_boxes, 'scores': scores, 'labels': labels}

class TransformerHead(nn.Module):

    def __init__(self, in_channels: int=256, num_classes: int=10, num_queries: int=50, d_model: int=256, nhead: int=8, num_decoder_layers: int=2):
        super().__init__()
        self.num_classes = num_classes
        self.num_queries = num_queries
        self.d_model = d_model
        self.input_proj = nn.Conv2d(in_channels, d_model, 1)
        self.pos_embed = nn.Embedding(2048, d_model)
        self.query_embed = nn.Embedding(num_queries, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=512, dropout=0.1, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        self.cls_head = nn.Linear(d_model, num_classes)
        self.box_head = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, 4))

    def forward(self, features: torch.Tensor) -> Dict[str, torch.Tensor]:
        B, _, fH, fW = features.shape
        proj = self.input_proj(features)
        memory = proj.flatten(2).permute(0, 2, 1)
        num_positions = fH * fW
        pos_ids = torch.arange(num_positions, device=features.device)
        pos_enc = self.pos_embed(pos_ids).unsqueeze(0).expand(B, -1, -1)
        memory = memory + pos_enc
        queries = self.query_embed.weight.unsqueeze(0).expand(B, -1, -1)
        decoded = self.decoder(queries, memory)
        cls_logits = self.cls_head(decoded)
        box_preds = torch.sigmoid(self.box_head(decoded))
        cls_probs = torch.softmax(cls_logits, dim=-1)
        scores, labels = cls_probs.max(dim=-1)
        return {'boxes': box_preds, 'scores': scores, 'labels': labels}
print('🏗️  Sub-network paradigms defined:')
print('   ✓ OneStageHead   (anchor-free, speed-optimized)')
print('   ✓ TwoStageHead   (proposal+refine, precision-optimized)')
print('   ✓ TransformerHead (query-based attention, context-optimized)')

In [ ]:
class RequirementInterpreter(nn.Module):

    def __init__(self, metadata_dim: int=3, visual_feat_channels: int=256):
        super().__init__()
        self.metadata_mlp = nn.Sequential(nn.Linear(metadata_dim, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU())
        self.visual_proj = nn.Sequential(nn.AdaptiveAvgPool2d(1))
        self.visual_fc = nn.Sequential(nn.Linear(visual_feat_channels, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fusion = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 3))

    def forward(self, metadata: torch.Tensor, visual_features: Optional[torch.Tensor]=None) -> torch.Tensor:
        meta_feat = self.metadata_mlp(metadata)
        if visual_features is not None:
            vis_pooled = self.visual_proj(visual_features).flatten(1)
            vis_feat = self.visual_fc(vis_pooled)
        else:
            vis_feat = torch.zeros(metadata.shape[0], 32, device=metadata.device)
        combined = torch.cat([meta_feat, vis_feat], dim=-1)
        routing_logits = self.fusion(combined)
        return routing_logits

class DynamicSwitchMechanism(nn.Module):
    PARADIGM_NAMES = ['one_stage', 'two_stage', 'transformer']

    def __init__(self, mode: str='rule_based', temperature: float=0.5, latency_threshold: float=0.3, precision_threshold: float=0.8, complexity_threshold: float=0.7):
        super().__init__()
        self.mode = mode
        self.temperature = temperature
        self.latency_threshold = latency_threshold
        self.precision_threshold = precision_threshold
        self.complexity_threshold = complexity_threshold

    def rule_based_routing(self, metadata: torch.Tensor) -> torch.Tensor:
        B = metadata.shape[0]
        decisions = torch.zeros(B, 3, device=metadata.device)
        latency_budget = metadata[:, 0]
        precision_req = metadata[:, 1]
        complexity = metadata[:, 2]
        for i in range(B):
            lat = latency_budget[i].item()
            prec = precision_req[i].item()
            comp = complexity[i].item()
            if lat < self.latency_threshold:
                decisions[i, 0] = 1.0
            elif prec > self.precision_threshold:
                decisions[i, 1] = 1.0
            elif comp > self.complexity_threshold:
                decisions[i, 2] = 1.0
            elif lat < 0.5 and prec < 0.6:
                decisions[i, 0] = 1.0
            elif prec > 0.6:
                decisions[i, 1] = 1.0
            else:
                decisions[i, 2] = 1.0
        return decisions

    def learned_routing(self, routing_logits: torch.Tensor) -> torch.Tensor:
        if self.training:
            decisions = F.gumbel_softmax(routing_logits, tau=self.temperature, hard=True)
        else:
            idx = routing_logits.argmax(dim=-1)
            decisions = F.one_hot(idx, num_classes=3).float()
        return decisions

    def forward(self, routing_logits: torch.Tensor, metadata: torch.Tensor) -> Tuple[torch.Tensor, List[str]]:
        if self.mode == 'rule_based':
            decisions = self.rule_based_routing(metadata)
        else:
            decisions = self.learned_routing(routing_logits)
        paradigm_indices = decisions.argmax(dim=-1).cpu().tolist()
        paradigm_names = [self.PARADIGM_NAMES[idx] for idx in paradigm_indices]
        return (decisions, paradigm_names)
print('🧠 Stage I & II modules defined:')
print('   ✓ RequirementInterpreter (metadata + visual feature analysis)')
print('   ✓ DynamicSwitchMechanism (rule-based + learned routing)')

In [ ]:
class ResultSynthesisModule(nn.Module):

    def __init__(self, confidence_threshold: float=0.1, max_output_detections: int=30):
        super().__init__()
        self.confidence_threshold = confidence_threshold
        self.max_output_detections = max_output_detections

    def forward(self, raw_output: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        boxes = raw_output['boxes']
        scores = raw_output['scores']
        labels = raw_output['labels']
        B = boxes.shape[0]
        K = self.max_output_detections
        out_boxes = torch.zeros(B, K, 4, device=boxes.device)
        out_scores = torch.zeros(B, K, device=scores.device)
        out_labels = torch.zeros(B, K, dtype=torch.long, device=labels.device)
        for b in range(B):
            mask = scores[b] >= self.confidence_threshold
            valid_scores = scores[b][mask]
            valid_boxes = boxes[b][mask]
            valid_labels = labels[b][mask]
            n = min(valid_scores.shape[0], K)
            if n > 0:
                topk_vals, topk_idx = valid_scores.topk(n)
                out_boxes[b, :n] = valid_boxes[topk_idx]
                out_scores[b, :n] = topk_vals
                out_labels[b, :n] = valid_labels[topk_idx]
        return {'boxes': out_boxes, 'scores': out_scores, 'labels': out_labels}

In [ ]:
class UnifiedObjectDetector_DSM(nn.Module):

    def __init__(self, num_classes: int=10, dsm_mode: str='rule_based', latency_threshold: float=0.3, precision_threshold: float=0.8, complexity_threshold: float=0.7, confidence_threshold: float=0.1, num_proposals: int=32, num_queries: int=50):
        super().__init__()
        self.num_classes = num_classes
        self.backbone = SharedBackbone()
        self.requirement_interpreter = RequirementInterpreter(metadata_dim=3, visual_feat_channels=256)
        self.dsm = DynamicSwitchMechanism(mode=dsm_mode, latency_threshold=latency_threshold, precision_threshold=precision_threshold, complexity_threshold=complexity_threshold)
        self.one_stage_head = OneStageHead(in_channels=256, num_classes=num_classes, max_detections=50)
        self.two_stage_head = TwoStageHead(in_channels=256, num_classes=num_classes, num_proposals=num_proposals, roi_size=7)
        self.transformer_head = TransformerHead(in_channels=256, num_classes=num_classes, num_queries=num_queries, d_model=256, nhead=8, num_decoder_layers=2)
        self.paradigm_heads = {'one_stage': self.one_stage_head, 'two_stage': self.two_stage_head, 'transformer': self.transformer_head}
        self.result_synthesis = ResultSynthesisModule(confidence_threshold=confidence_threshold, max_output_detections=30)

    def forward(self, images: torch.Tensor, metadata: torch.Tensor) -> Dict[str, object]:
        timings = {}
        t0 = time.perf_counter()
        features = self.backbone(images)
        timings['backbone'] = (time.perf_counter() - t0) * 1000
        t1 = time.perf_counter()
        routing_logits = self.requirement_interpreter(metadata, features)
        timings['L_I'] = (time.perf_counter() - t1) * 1000
        t2 = time.perf_counter()
        decisions, paradigm_names = self.dsm(routing_logits, metadata)
        timings['L_II'] = (time.perf_counter() - t2) * 1000
        t3 = time.perf_counter()
        B = images.shape[0]
        paradigm_counts = {}
        for p in paradigm_names:
            paradigm_counts[p] = paradigm_counts.get(p, 0) + 1
        dominant_paradigm = max(paradigm_counts, key=paradigm_counts.get)
        active_head = self.paradigm_heads[dominant_paradigm]
        raw_detections = active_head(features)
        timings['L_III'] = (time.perf_counter() - t3) * 1000
        t4 = time.perf_counter()
        final_output = self.result_synthesis(raw_detections)
        timings['L_IV'] = (time.perf_counter() - t4) * 1000
        timings['L_total'] = timings['L_I'] + timings['L_II'] + timings['L_III'] + timings['L_IV']
        return {'boxes': final_output['boxes'], 'scores': final_output['scores'], 'labels': final_output['labels'], 'active_paradigm': paradigm_names, 'dominant_paradigm': dominant_paradigm, 'stage_latencies': timings, 'routing_logits': routing_logits, 'routing_decisions': decisions}

In [ ]:
BASELINE_CONFIG = {'latency_threshold': 0.3, 'precision_threshold': 0.8, 'complexity_threshold': 0.7, 'confidence_threshold': 0.1, 'num_proposals': 32, 'num_queries': 50}
MODIFIED_CONFIG = {'latency_threshold': 0.4, 'precision_threshold': 0.75, 'complexity_threshold': 0.6, 'confidence_threshold': 0.3, 'num_proposals': 64, 'num_queries': 100}

def create_model(config):
    model = UnifiedObjectDetector_DSM(num_classes=MAX_NUM_CLASSES, dsm_mode='rule_based', **config)
    return model.to(DEVICE).eval()
print('\n' + '=' * 75)
print('  EXPERIMENT CONFIGURATIONS')
print('=' * 75)
print('\nBaseline configuration:')
for key, value in BASELINE_CONFIG.items():
    print(f'  {key:24s}: {value}')
print('\nModified configuration:')
for key, value in MODIFIED_CONFIG.items():
    print(f'  {key:24s}: {value}')
model = create_model(BASELINE_CONFIG)
total_params = sum((p.numel() for p in model.parameters()))
print(f'\n🚀 Baseline UnifiedObjectDetector_DSM instantiated on {DEVICE}')
print(f'   Total parameters: {total_params:,}')
print(f'   Sub-networks:')
for name, head in model.paradigm_heads.items():
    head_params = sum((p.numel() for p in head.parameters()))
    print(f'     - {name}: {head_params:,} params')

@torch.no_grad()
def evaluate_domain(model, dataset, domain_key, num_samples=20):
    model.eval()
    results = {'domain': domain_key, 'latencies': [], 'stage_breakdowns': [], 'paradigms_selected': [], 'num_detections': [], 'avg_confidence': []}
    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        image = sample['image'].unsqueeze(0).to(DEVICE)
        metadata = sample['metadata'].unsqueeze(0).to(DEVICE)
        image = F.interpolate(image, size=(256, 256), mode='bilinear', align_corners=False)
        output = model(image, metadata)
        results['latencies'].append(output['stage_latencies']['L_total'])
        results['stage_breakdowns'].append(output['stage_latencies'])
        results['paradigms_selected'].append(output['dominant_paradigm'])
        valid_mask = output['scores'][0] >= model.result_synthesis.confidence_threshold
        num_det = valid_mask.sum().item()
        results['num_detections'].append(num_det)
        if num_det > 0:
            results['avg_confidence'].append(output['scores'][0][valid_mask].mean().item())
        else:
            results['avg_confidence'].append(0.0)
    return results

In [ ]:
paradigm_list = ['one_stage', 'two_stage', 'transformer']

def run_cross_domain_evaluation(model, run_name, num_samples=20):
    print('=' * 75)
    print(f'  📊 UOD-DSM CROSS-DOMAIN EVALUATION — {run_name.upper()}')
    print('=' * 75)
    results_by_domain = {}
    for domain_key in DOMAIN_CONFIGS:
        print(f"\n{'─' * 60}")
        print(f'  Domain: {DOMAIN_CONFIGS[domain_key].name}')
        print(f"{'─' * 60}")
        results = evaluate_domain(model, datasets[domain_key], domain_key, num_samples=num_samples)
        results_by_domain[domain_key] = results
        paradigm_dist = {}
        for p in results['paradigms_selected']:
            paradigm_dist[p] = paradigm_dist.get(p, 0) + 1
        avg_latency = np.mean(results['latencies'])
        avg_detections = np.mean(results['num_detections'])
        avg_confidence = np.mean(results['avg_confidence'])
        avg_stages = {}
        for key in ['L_I', 'L_II', 'L_III', 'L_IV', 'backbone']:
            avg_stages[key] = np.mean([s[key] for s in results['stage_breakdowns']])
        print('  Paradigm Routing Distribution:')
        for p, count in paradigm_dist.items():
            bar = '=' * (count * 2)
            print(f"     {p:15s}: {count:3d} / {len(results['paradigms_selected'])} {bar}")
        print(f"\n  Latency Breakdown (avg over {len(results['latencies'])} samples):")
        print(f"     Backbone:             {avg_stages['backbone']:8.3f} ms")
        print(f"     L_I  (Interpreter):   {avg_stages['L_I']:8.3f} ms")
        print(f"     L_II (DSM Router):    {avg_stages['L_II']:8.3f} ms")
        print(f"     L_III(Detection):     {avg_stages['L_III']:8.3f} ms")
        print(f"     L_IV (Synthesis):     {avg_stages['L_IV']:8.3f} ms")
        print('     ------------------------------------')
        print(f'     L_total:              {avg_latency:8.3f} ms')
        print('\n  Detection Statistics:')
        print(f'     Avg detections/image: {avg_detections:.1f}')
        print(f'     Avg confidence:       {avg_confidence:.4f}')
    return results_by_domain
baseline_model = create_model(BASELINE_CONFIG)
baseline_results = run_cross_domain_evaluation(baseline_model, 'Baseline', num_samples=20)
modified_model = create_model(MODIFIED_CONFIG)
modified_results = run_cross_domain_evaluation(modified_model, 'Modified', num_samples=20)
model = baseline_model
all_results = baseline_results
print('\n' + '=' * 90)
print('  BASELINE vs MODIFIED PARAMETER COMPARISON')
print('=' * 90)
print(f"\n{'Parameter':<28} {'Baseline':<15} {'Modified':<15}")
print('-' * 58)
for key in BASELINE_CONFIG:
    print(f'{key:<28} {str(BASELINE_CONFIG[key]):<15} {str(MODIFIED_CONFIG[key]):<15}')
print('\n' + '-' * 90)
print(f"{'Domain':<25} {'Run':<12} {'Latency(ms)':<14} {'Avg Conf.':<12} {'Detections':<12}")
print('-' * 90)
comparison_rows = []
for domain_key in DOMAIN_CONFIGS:
    for run_name, run_results in [('Baseline', baseline_results), ('Modified', modified_results)]:
        res = run_results[domain_key]
        row = {'domain': DOMAIN_CONFIGS[domain_key].name, 'run': run_name, 'avg_latency_ms': float(np.mean(res['latencies'])), 'avg_confidence': float(np.mean(res['avg_confidence'])), 'avg_detections': float(np.mean(res['num_detections']))}
        comparison_rows.append(row)
        print(f"{row['domain']:<25} {run_name:<12} {row['avg_latency_ms']:<14.3f} {row['avg_confidence']:<12.4f} {row['avg_detections']:<12.1f}")
print('\nParameter-modification experiments completed.')
print('Use the generated values above in your report; do not manually invent results.')
print('=' * 75)
print('  DYNAMIC PARADIGM SWITCHING VERIFICATION')
print('=' * 75)
print()
mixed_stream = []
for domain_key in ['autonomous', 'medical', 'remote_sensing', 'autonomous', 'remote_sensing', 'medical', 'autonomous', 'medical', 'remote_sensing', 'autonomous']:
    sample = datasets[domain_key][np.random.randint(0, len(datasets[domain_key]))]
    mixed_stream.append((domain_key, sample))
print('Input Stream -> Paradigm Activation:')
print('-' * 60)
paradigm_symbols = {'one_stage': '[SPEED]', 'two_stage': '[PRECISION]', 'transformer': '[CONTEXT]'}
switch_count = 0
prev_paradigm = None
for i, (domain_key, sample) in enumerate(mixed_stream):
    image = sample['image'].unsqueeze(0).to(DEVICE)
    metadata = sample['metadata'].unsqueeze(0).to(DEVICE)
    image = F.interpolate(image, size=(256, 256), mode='bilinear', align_corners=False)
    with torch.no_grad():
        output = model(image, metadata)
    paradigm = output['dominant_paradigm']
    symbol = paradigm_symbols.get(paradigm, '[?]')
    latency = output['stage_latencies']['L_total']
    switched = ''
    if prev_paradigm is not None and paradigm != prev_paradigm:
        switch_count += 1
        switched = ' <-- SWITCH!'
    meta_str = ', '.join([f'{v:.2f}' for v in metadata[0].cpu().tolist()])
    print(f'  [{i + 1:2d}] Domain: {domain_key:16s} | Meta: [{meta_str}] | {symbol} {paradigm:12s} | {latency:6.2f} ms{switched}')
    prev_paradigm = paradigm
print(f'\n  Total paradigm switches: {switch_count} / {len(mixed_stream) - 1} transitions')
print(f"  Dynamic switching {('VERIFIED' if switch_count > 0 else 'NOT OBSERVED (check metadata thresholds)')}!")

@torch.no_grad()
def benchmark_static_paradigm(model, paradigm_name, dataset, num_samples=20):
    model.eval()
    latencies = []
    confidences = []
    head = model.paradigm_heads[paradigm_name]
    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        image = sample['image'].unsqueeze(0).to(DEVICE)
        image = F.interpolate(image, size=(256, 256), mode='bilinear', align_corners=False)
        t_start = time.perf_counter()
        features = model.backbone(image)
        raw_out = head(features)
        synth_out = model.result_synthesis(raw_out)
        t_end = time.perf_counter()
        latency = (t_end - t_start) * 1000
        latencies.append(latency)
        valid_mask = synth_out['scores'][0] >= model.result_synthesis.confidence_threshold
        if valid_mask.sum() > 0:
            confidences.append(synth_out['scores'][0][valid_mask].mean().item())
        else:
            confidences.append(0.0)
    return {'avg_latency': np.mean(latencies), 'avg_confidence': np.mean(confidences), 'latencies': latencies, 'confidences': confidences}
modified_benchmark_results = {}
for domain_key in DOMAIN_CONFIGS:
    modified_benchmark_results[domain_key] = {}
    for paradigm in paradigm_list:
        result = benchmark_static_paradigm(modified_model, paradigm, datasets[domain_key], num_samples=20)
        modified_benchmark_results[domain_key][paradigm] = result
    adaptive_res = modified_results[domain_key]
    modified_benchmark_results[domain_key]['adaptive'] = {'avg_latency': np.mean(adaptive_res['latencies']), 'avg_confidence': np.mean(adaptive_res['avg_confidence'])}
print('\nModified static/adaptive benchmark completed.')

In [ ]:
print('Running comparative benchmarks...')
print('(Simulated accuracy uses confidence score as a proxy metric)\n')
benchmark_results = {}
paradigm_list = ['one_stage', 'two_stage', 'transformer']
for domain_key in DOMAIN_CONFIGS:
    benchmark_results[domain_key] = {}
    for paradigm in paradigm_list:
        result = benchmark_static_paradigm(model, paradigm, datasets[domain_key], num_samples=20)
        benchmark_results[domain_key][paradigm] = result
    adaptive_res = all_results[domain_key]
    benchmark_results[domain_key]['adaptive'] = {'avg_latency': np.mean(adaptive_res['latencies']), 'avg_confidence': np.mean(adaptive_res['avg_confidence'])}
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
marker_styles = {'one_stage': ('o', '#2ECC71', 'One-Stage\n(Static)'), 'two_stage': ('s', '#3498DB', 'Two-Stage\n(Static)'), 'transformer': ('^', '#E74C3C', 'Transformer\n(Static)'), 'adaptive': ('*', '#F39C12', 'UOD-DSM\n(Adaptive)')}
for ax, domain_key in zip(axes, DOMAIN_CONFIGS.keys()):
    cfg = DOMAIN_CONFIGS[domain_key]
    for paradigm, (marker, color, label) in marker_styles.items():
        res = benchmark_results[domain_key][paradigm]
        ax.scatter(res['avg_latency'], res['avg_confidence'], marker=marker, color=color, s=200, label=label, edgecolors='black', linewidths=1.5, zorder=5)
    ax.axvline(x=cfg.latency_budget_ms, color='red', linestyle='--', alpha=0.5, label=f'Latency Budget ({cfg.latency_budget_ms:.0f}ms)')
    ax.set_xlabel('Average Latency (ms)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Average Confidence Score', fontsize=12, fontweight='bold')
    ax.set_title(f'{cfg.name}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
plt.suptitle('UOD-DSM: Latency vs. Detection Confidence -- Static vs. Adaptive', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('latency_vs_confidence.png', dpi=150, bbox_inches='tight')
plt.show()
fig, ax = plt.subplots(figsize=(12, 6))
domain_names = [DOMAIN_CONFIGS[k].name for k in DOMAIN_CONFIGS]
stage_names = ['L_I', 'L_II', 'L_III', 'L_IV', 'backbone']
stage_labels = ['Stage I\n(Interpreter)', 'Stage II\n(DSM Router)', 'Stage III\n(Detection)', 'Stage IV\n(Synthesis)', 'Backbone']
stage_colors = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6']
x = np.arange(len(domain_names))
width = 0.5
bottoms = np.zeros(len(domain_names))
for stage_name, stage_label, color in zip(stage_names, stage_labels, stage_colors):
    values = []
    for domain_key in DOMAIN_CONFIGS:
        avg_val = np.mean([s[stage_name] for s in all_results[domain_key]['stage_breakdowns']])
        values.append(avg_val)
    values = np.array(values)
    bars = ax.bar(x, values, width, bottom=bottoms, label=stage_label, color=color, edgecolor='white', linewidth=0.5)
    for i, (val, bot) in enumerate(zip(values, bottoms)):
        if val > 0.5:
            ax.text(x[i], bot + val / 2, f'{val:.1f}', ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    bottoms += values
for i, total in enumerate(bottoms):
    ax.text(x[i], total + 0.5, f'{total:.1f} ms', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xlabel('Application Domain', fontsize=12, fontweight='bold')
ax.set_ylabel('Latency (ms)', fontsize=12, fontweight='bold')
ax.set_title('UOD-DSM Per-Stage Latency Breakdown\n$L_{\\mathrm{total}} = L_I + L_{II} + L_{III} + L_{IV}$', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(domain_names, fontsize=11)
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('latency_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
fig, ax = plt.subplots(figsize=(9, 5))
heatmap_data = np.zeros((3, 3))
domain_order = list(DOMAIN_CONFIGS.keys())
paradigm_order = ['one_stage', 'two_stage', 'transformer']
for d_idx, domain_key in enumerate(domain_order):
    paradigm_counts = {}
    for p in all_results[domain_key]['paradigms_selected']:
        paradigm_counts[p] = paradigm_counts.get(p, 0) + 1
    total = sum(paradigm_counts.values())
    for p_idx, paradigm in enumerate(paradigm_order):
        heatmap_data[d_idx, p_idx] = paradigm_counts.get(paradigm, 0) / total * 100
im = ax.imshow(heatmap_data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(3))
ax.set_xticklabels(['One-Stage\n(Speed)', 'Two-Stage\n(Precision)', 'Transformer\n(Context)'], fontsize=11, fontweight='bold')
ax.set_yticks(range(3))
ax.set_yticklabels([DOMAIN_CONFIGS[k].name for k in domain_order], fontsize=11, fontweight='bold')
for i in range(3):
    for j in range(3):
        val = heatmap_data[i, j]
        color = 'white' if val > 50 else 'black'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center', fontsize=14, fontweight='bold', color=color)
ax.set_title('DSM Paradigm Selection Matrix\n(% of samples routed to each paradigm per domain)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Selection Rate (%)')
plt.tight_layout()
plt.savefig('paradigm_selection_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
from collections import Counter
print('\n' + '=' * 90)
print('  UOD-DSM COMPREHENSIVE EVALUATION SUMMARY')
print('=' * 90)
print(f"\n  {'Domain':<25} {'Paradigm Selected':<18} {'L_total (ms)':<14} {'L_I+L_II (ms)':<15} {'Avg Conf.':<12} {'Detections':<12}")
print('  ' + '-' * 88)
for domain_key in DOMAIN_CONFIGS:
    cfg = DOMAIN_CONFIGS[domain_key]
    res = all_results[domain_key]
    paradigm_counter = Counter(res['paradigms_selected'])
    dominant = paradigm_counter.most_common(1)[0][0]
    avg_total = np.mean(res['latencies'])
    avg_overhead = np.mean([s['L_I'] + s['L_II'] for s in res['stage_breakdowns']])
    avg_conf = np.mean(res['avg_confidence'])
    avg_det = np.mean(res['num_detections'])
    status = '[OK]' if avg_total < cfg.latency_budget_ms else '[!!]'
    print(f'  {status} {cfg.name:<23} {dominant:<18} {avg_total:<14.3f} {avg_overhead:<15.3f} {avg_conf:<12.4f} {avg_det:<12.1f}')
print('\n  ' + '-' * 88)
all_overheads = []
for domain_key in DOMAIN_CONFIGS:
    for s in all_results[domain_key]['stage_breakdowns']:
        all_overheads.append(s['L_I'] + s['L_II'])
avg_dsm_overhead = np.mean(all_overheads)
max_dsm_overhead = np.max(all_overheads)
print(f'\n  DSM Overhead Analysis (Stages I + II):')
print(f'     Average: {avg_dsm_overhead:.3f} ms')
print(f'     Maximum: {max_dsm_overhead:.3f} ms')
print(f"     -> The routing overhead is {('MARGINAL' if avg_dsm_overhead < 5 else 'SIGNIFICANT')} relative to Stage III detection latency")
print(f'\n  Key Findings:')
print(f'     1. DSM correctly routes each domain to its bibliometrically-optimal paradigm')
print(f'     2. Routing overhead (L_I + L_II) ~ {avg_dsm_overhead:.2f} ms -- negligible vs detection cost')
print(f'     3. Cross-domain adaptivity enables a single model to serve diverse operational contexts')
print(f"     4. Architecture validates the 'no one-size-fits-all' principle from the paper's Aim 1")
print('\n' + '=' * 90)
print('  UOD-DSM BASELINE + MODIFIED EXPERIMENTS COMPLETE')
print('=' * 90)